In [1]:
import os
import torch
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder 

# ===========================
# 1. 准备数据
# ===========================
print(">>> 1. 正在加载与切分文档...")
pdf_path = "./data/stat_book.pdf" 
if not os.path.exists(pdf_path):
    print(f"找不到文件 {pdf_path}，请检查路径！")
    exit()

loader = PyPDFLoader(pdf_path)
docs = loader.load()

# 切分策略：稍微改小一点 chunk_size
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
all_splits = text_splitter.split_documents(docs)
# 过滤空内容
all_splits = [d for d in all_splits if d.page_content.strip()] 

# ===========================
# 2. 基础检索 (Retriever)
# ===========================
print(">>> 2. 构建基础向量索引...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")
vector_store = FAISS.from_documents(all_splits, embeddings)

# ===========================
# 3. 高级检索 (手动实现 Rerank)
# ===========================
print(">>> 3. 加载 Rerank 模型 (第一次运行会下载)...")
reranker_model = CrossEncoder('BAAI/bge-reranker-base', max_length=512)

def retrieve_and_rerank(query, top_k=3):
    """
    两阶段检索函数
    """
    print(f"\n[检索中] 第一阶段：向量召回 Top-10...")
    # 1. 初筛
    candidates = vector_store.similarity_search(query, k=10)
    
    # 2. 准备打分数据：[(query, doc1), (query, doc2)...]
    pairs = [[query, doc.page_content] for doc in candidates]
    
    # 3. 计算分数 (使用 predict 方法)
    print(f"[检索中] 第二阶段：模型打分重排序...")
    scores = reranker_model.predict(pairs)
    
    # 4. 排序并结合
    # 把分数和文档绑在一起，按分数从高到低排序
    combined = list(zip(candidates, scores))
    sorted_docs = sorted(combined, key=lambda x: x[1], reverse=True)
    
    # 5. 取前几名
    final_docs = [doc for doc, score in sorted_docs[:top_k]]
    
    # 打印前三名的分数看看
    print("--- 最终精选片段得分 ---")
    for i, (doc, score) in enumerate(sorted_docs[:top_k]):
        print(f"Rank {i+1}: Score={score:.4f} | 内容: {doc.page_content[:20]}...")
        
    return final_docs

# ===========================
# 4. 问答链
# ===========================
API_KEY = "sk-43401194122e4d4f8645889b070af0df" 
BASE_URL = "https://api.deepseek.com"

llm = ChatOpenAI(model="deepseek-chat", api_key=API_KEY, base_url=BASE_URL, temperature=0.1)

prompt = ChatPromptTemplate.from_template("""
基于以下精选信息回答问题。

【已知信息】:
{context}

【问题】:
{question}
""")

chain = prompt | llm | StrOutputParser()

# ===========================
# 测试
# ===========================
q = "请总结这篇文档的核心内容"

try:
    # 1. 获取重排序后的文档
    best_docs = retrieve_and_rerank(q)
    
    # 2. 拼装 context
    context = "\n\n".join([d.page_content for d in best_docs])
    
    # 3. 问大模型
    print(f"\n[思考中] 正在生成回答...")
    answer = chain.invoke({"context": context, "question": q})
    
    print("\n=== AI 最终回答 ===")
    print(answer)
except Exception as e:
    print(f"出错: {e}")

>>> 1. 正在加载与切分文档...
>>> 2. 构建基础向量索引...


C:\Users\38244\AppData\Local\Temp\ipykernel_40380\3087696888.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


>>> 3. 加载 Rerank 模型 (第一次运行会下载)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[检索中] 第一阶段：向量召回 Top-10...
[检索中] 第二阶段：模型打分重排序...
--- 最终精选片段得分 ---
Rank 1: Score=0.0152 | 内容: and a proper network...
Rank 2: Score=0.0022 | 内容: determine a dynamica...
Rank 3: Score=0.0013 | 内容: 2017). In this conte...

[思考中] 正在生成回答...

=== AI 最终回答 ===
这篇文档的核心内容可以总结为以下几点：

1. **神经网络逼近理论**：文档讨论了如何通过适当初始化网络，使得一个n维动力系统能够被嵌入到一个更高维度的系统中，并证明存在一个具有N个隐藏单元的循环神经网络（RNN），其轨迹在时间区间I = [0, T]上能以任意精度ϵ逼近原系统在紧集K上的轨迹。

2. **LTC与CT-RNN的对比**：文中特别提到了LTC（Liquid Time-Constant Networks）与CT-RNN（Continuous-Time RNN）在普适性证明上的根本区别，暗示LTC在表达动力系统方面可能具有独特的理论优势。

3. **深度模型的可视化分析**：文档还涉及了如何通过可视化技术（例如对网络激活进行主成分分析PCA）来理解深度模型如何将简单的输入轨迹（如二维圆形输入）逐步转换为更复杂的模式。

**核心主旨**：文档聚焦于从理论（动力系统逼近）和实证（激活模式分析）两个层面，探讨特定类型的循环神经网络（尤其是LTC）在建模和表达复杂动态行为方面的能力与特性。
